# Первый этап: сбор цифровых следов переоценки ВО и СПО

**Три источника:** официальные Telegram-каналы Минпросвещения и Минобрнауки, публичные обсуждения Пикабу, вопросы и ответы «Табитуриента».

**Период публикаций:** 01.01.2018–06.10.2026 включительно, по московскому времени. Периодизация: 2018; 2019–2021; 2022–2026. Внутри второго этапа 2019 — допандемийный год, 2020–2021 — пандемийные годы. При последующем анализе их следует разделять.

Исходные документы: [исследовательский бриф](https://docs.google.com/document/d/1CoG8eakAeDvc_yytix-rmGgn6U09XQtnUfxVUEBjMuk/edit) и [требования преподавателя](https://docs.google.com/document/d/1Q6FDDf87wQtC3-ADtgcZIBB-ukvj-v6AXy4f21X8mI8/edit). Последующие решения команды имеют приоритет: расширенный период, срез по 06.10.2026, тематическая выборка вместо полного архива.

Это корпус **кандидатов для содержательного кодирования**. Совпадение слов не доказывает охлаждение притязаний, а число найденных текстов не измеряет популярность мнения в обществе. Сохраняются и критика, и защита ВО, и положительные аргументы о СПО.

## План до кода

1. Импортировать библиотеки; задать период, ключевые слова и режим запуска. Прочитать готовую выборку из Excel рядом с notebook. В демонстрации взять до 20 родительских публикаций каждого источника, равномерно разнесённых по сохранённому списку.
2. Открывать публичные страницы через Selenium с явным ожиданием элемента. BeautifulSoup разбирает только полученный `page_source`; pandas формирует таблицы. Прямых API/AJAX-запросов и работы с аккаунтами нет. Функции общих проверок нормализуют счётчики и московские даты; журнал фиксирует ошибки.
3. Для Telegram прочитать карточки `data-post`: дату публикации, полный текст, просмотры, реакции, медиа, признак пересылки. Альбом — одна строка. Для повторения выборки использовать сохранённые URL; для расширения — листать всю веб-ленту назад с `before` и применять словарь локально. Не использовать неполную поисковую выдачу как архив.
4. Для Пикабу прочитать полный пост, заголовок, теги, дату, рейтинг и число комментариев. Сохранить только первоначально загруженные комментарии с известной датой внутри периода, связав их с постом. Для расширения собирать ссылки из теговых лент по страницам. Отбор требует образовательного контекста и слов выбора, оценки или препятствий.
5. Для «Табитуриента» прочитать вопрос и каждый ответ отдельной строкой с общим `question_id`. Взять скрытое продолжение длинного текста из DOM, удалить только надпись «Показать полностью». Отдельно сохранить метку приёмной комиссии и счётчик вложенных комментариев; сами вложенные комментарии не подгружать. В режиме расширения обойти ID до максимального, видимого в списке.
6. Сформировать три DataFrame, проверить уникальность ключей и столбцов, даты и связи ответов с родительскими сообщениями. Добавить год, этап, хэштеги и вспомогательный признак тематического контекста. Этот признак не является содержательным кодом охлаждения.
7. Сохранить таблицы в один Excel с тремя листами и в CSV. Показать только объяснённые сводки: объём, виды сообщений и границы дат. Демонстрация сохраняется отдельно и не заменяет готовую исследовательскую выборку.

## Как запустить

Положить этот notebook и `База_трёх_источников.xlsx` в одну папку. Нужны Chrome и перечисленные ниже пакеты. Команда установки приведена только как справка и автоматически не выполняется:

```text
pip install selenium pandas beautifulsoup4 openpyxl
```

`Kernel → Restart & Run All` запускает демонстрацию. `FULL_RUN = True` повторно читает всю сохранённую выборку. Дополнительно `EXPAND_ARCHIVE = True` включает более долгий обход архивов; для Пикабу предел страниц задаётся явно. Для длительного сбора с возобновлением есть отдельные скрипты в приложенном архиве проекта. Все функции, нужные этому notebook, находятся ниже: внешние модули проекта ему не требуются.

При изменении вёрстки или проверке доступа ошибка остаётся видимой. Нельзя гарантировать вечную неизменность сайтов; конкретные селекторы проверены живыми загрузками 07.10.2026.

In [ ]:
# 1. Импорт, параметры и чтение готовой тематической выборки
import csv, json, os, re, time
from copy import copy
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException, InvalidSessionIdException
from IPython.display import display

FULL_RUN = False
EXPAND_ARCHIVE = False
N_DEMO = 20
START, END = '2018-01-01', '2026-10-06'
MAX_TAG_PAGES = 500  # Явный предел расширения, а не обещание полного архива.
PAUSE = 2
MOSCOW = timezone(timedelta(hours=3))
ERROR_DIR = Path('logs')
saved = pd.read_excel('База_трёх_источников.xlsx', sheet_name=None)
config = dict(start=START, end=END)

def take_sample(items):
    if FULL_RUN or len(items) <= N_DEMO:
        return items
    if N_DEMO <= 1:
        return items[:max(0,N_DEMO)]
    return [items[round(i*(len(items)-1)/(N_DEMO-1))] for i in range(N_DEMO)]
TERMS = ['колледж', 'техникум', 'СПО', 'ПТУ', 'высшее образован', 'вуз', 'университет', 'Профессионалитет', 'рабочи',
 'професси', 'профориентац', 'поступ', 'абитуриент', '10 класс', '10-й класс', 'десятый класс', '9 класс',
 '9-го класса', 'ОГЭ', 'ЕГЭ', 'родител', 'диплом', 'престиж', 'трудоустрой', 'зарплат', 'бюджет', 'платное',
 'не берут', 'не приняли', 'отказ', 'передум', 'не пройду', 'не прош', 'попроще', 'перевод', 'перевестись',
 'WorldSkills', 'EuroSkills', 'Молодые профессионалы']
config['terms'] = TERMS
# Вывод: размер готовой выборки, которую будем проверять повторным чтением сайтов.
display(pd.DataFrame({'источник': saved.keys(), 'строк в готовой базе': [len(x) for x in saved.values()]}))

## 2. Общие операции

Драйвер не использует личный профиль. Отсутствующее число сохраняется как пропуск; `0` остаётся нулём. Обозначение `1.54K` преобразуется в приблизительное число 1540, а исходное отображение сохраняется отдельным полем. Изменение часового пояса учитывается до сравнения с границей 06.10.2026. Журнал ошибок служит для проверки пропусков.

In [ ]:
# 2. Функции: запуск браузера, ожидание, нормализация и журнал
def create_driver():
    """На другом компьютере Selenium сам подберёт драйвер установленного Chrome."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--window-size=1440,1000')
    options.page_load_strategy = 'eager'
    options.add_experimental_option('prefs', {'download_restrictions': 3})
    service = Service(os.environ['CHROMEDRIVER']) if os.environ.get('CHROMEDRIVER') else Service()
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(45)
    return driver

def load_page(driver, url, selector):
    """Только обычная навигация Selenium; два повтора временного сбоя загрузки."""
    for attempt in range(3):
        try:
            driver.get(url)
            title = driver.title.lower()
            visible_start = driver.find_element(By.TAG_NAME,'body').text[:1000].lower()
            if any(x in title for x in ['privacy error', 'just a moment', 'access denied', 'проверка браузера']):
                raise PermissionError('Сайт требует проверки доступа или показывает ошибку сертификата')
            if any(x in visible_start for x in ['проверяем, что вы не робот','подтвердите, что вы человек','verify you are human','too many requests']):
                raise PermissionError('Сайт запросил проверку доступа или ограничил частоту запросов')
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
            if 'pikabu.ru/story/' in url:
                try:
                    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '.comments')))
                except TimeoutException:
                    note_error('pikabu', url, 'Блок комментариев не появился за 10 секунд; сохранён доступный DOM')
            return BeautifulSoup(driver.page_source, 'html.parser')
        except (TimeoutException, WebDriverException):
            try:
                title = driver.title.lower()
            except WebDriverException:
                raise InvalidSessionIdException('Браузер недоступен: повторите запуск; готовые страницы сохранены')
            if any(x in title for x in ['privacy error', 'just a moment', 'access denied', 'captcha']):
                raise PermissionError('Остановка на проверке доступа; обход защиты не выполняется')
            if attempt == 2:
                raise
            time.sleep(2 * (attempt + 1))

def parse_count(raw):
    """1.54K → 1540; округлённое обозначение сохраняется отдельным полем."""
    if raw is None or not str(raw).strip():
        return None
    text = str(raw).replace('\u2212', '-').replace('\xa0', '').replace(' ', '').strip()
    match = re.fullmatch(r'([+-]?\d+(?:[.,]\d+)?)([kкmм])?', text, re.I)
    if not match:
        return None
    value = float(match[1].replace(',', '.'))
    multiplier = {'k': 1000, 'к': 1000, 'm': 1000000, 'м': 1000000}.get((match[2] or '').lower(), 1)
    return round(value * multiplier)

def within_period(value, start, end):
    """Верхняя граница включает весь день по московскому времени."""
    if not value:
        return False
    date = datetime.fromisoformat(str(value).replace('Z', '+00:00'))
    if date.tzinfo:
        date = date.astimezone(MOSCOW)
    return start <= date.date().isoformat() <= end

def keyword_hits(text, terms):
    normalized = str(text).lower().replace('ё', 'е')
    hits = []
    for term in terms:
        word = term.lower().replace('ё', 'е')
        if word == 'вуз':
            pattern = r'(?<!\w)вуз\w*'
        else:
            pattern = r'(?<!\w)' + re.escape(word) + r'(?!\w)' if word in {'спо','во','пту','егэ','огэ'} else re.escape(word)
        if re.search(pattern, normalized):
            hits.append(term)
    return hits

def note_error(source, url, reason):
    ERROR_DIR.mkdir(parents=True, exist_ok=True)
    path = ERROR_DIR / f'{source}_errors.csv'
    exists = path.exists()
    with path.open('a', encoding='utf-8-sig', newline='') as stream:
        writer = csv.writer(stream)
        if not exists:
            writer.writerow(['collected_at', 'source', 'url', 'reason'])
        writer.writerow([datetime.now(MOSCOW).isoformat(timespec='seconds'), source, url, reason])

## 3. Telegram

**Единица:** пост; каналы различаются переменной `channel`. Комментарии Telegram не входят в базу. `is_forward` — факт пересылки, а не её количество. Отсутствие блока реакций означает, что значение неизвестно. Валентность эмодзи здесь не присваивается.

В исходной выборке: до 30 тематических постов на канал и год, начиная с 31 июля назад в пределах того же года; дополнительное окно заканчивается 06.10.2026. Это сезонная целевая выборка. Её нельзя использовать как знаменатель доли темы во всех сообщениях канала. Даты и причины остановки окон находятся в протоколе сбора. Для повторной проверки notebook читает те же публикации по сохранённым адресам.

In [ ]:
# 3.1. Одна карточка Telegram → словарь
def read_telegram_message(message):
    """Одна строка = один видимый пост; все карточки альбома остаются в этой строке."""
    channel, number = message["data-post"].split("/")
    date = message.select_one("a.tgme_widget_message_date time[datetime]")
    if date is None:
        raise ValueError("У сообщения отсутствует дата публикации")
    published = datetime.fromisoformat(date["datetime"]).astimezone(MOSCOW).isoformat()
    body = message.select_one(".tgme_widget_message_text")
    views = message.select_one(".tgme_widget_message_views")
    views_raw = views.get_text(strip=True) if views else None
    album = sorted({int(a["href"].split("?")[0].rstrip("/").split("/")[-1])
                    for a in message.select(".tgme_widget_message_grouped_wrap a[href]")
                    if re.search(r"/" + re.escape(channel) + r"/\d+\?single", a["href"])})
    number = min([int(number)] + album)
    # Reactions excluded by docs/branch-plan.md.
    reactions_raw, reactions = [], []
    media = {kind: len(message.select(selector)) for kind, selector in {
        "photo": ".tgme_widget_message_photo_wrap",
        "video": ".tgme_widget_message_video_player",
        "document": ".tgme_widget_message_document_wrap",
        "audio": ".tgme_widget_message_audio",
        "voice": ".tgme_widget_message_voice",
        "sticker": ".tgme_widget_message_sticker",
    }.items()}
    return dict(
        post_id=number, channel=channel, url=f"https://t.me/{channel}/{number}",
        published_at=published, text=body.get_text(" ", strip=True) if body else "",
        views_raw=views_raw, views=parse_count(views_raw),
        # Нет блока реакций ≠ нулевое число реакций.
        reactions_raw=json.dumps(reactions_raw, ensure_ascii=False) if reactions_raw else None,
        reactions=json.dumps(reactions, ensure_ascii=False) if reactions else None,
        media=";".join(kind for kind, count in media.items() if count),
        media_count=sum(media.values()), album_post_ids=json.dumps(album),
        is_forward=message.select_one(".tgme_widget_message_forwarded_from") is not None,
        collected_at=datetime.now(MOSCOW).isoformat(timespec="seconds"),
    )

In [ ]:
# 3.2. Цикл по сохранённым публикациям или по архиву канала
def telegram_table(driver):
    rows = {}
    def add(message, allowed=None):
        row = read_telegram_message(message)
        key = f"{row['channel']}/{row['post_id']}"
        hits = keyword_hits(row['text'], TERMS)
        if (allowed is None or key == allowed) and hits and within_period(row['published_at'], START, END):
            row.update(matched_terms='; '.join(hits), query_matches='', selection_method='archive' if EXPAND_ARCHIVE and FULL_RUN else 'saved_sample_reread')
            rows[key] = row

    if FULL_RUN and EXPAND_ARCHIVE:
        for channel in ['minprosrf', 'minobrnaukiofficial']:
            before = None
            while True:
                url = f'https://t.me/s/{channel}' + (f'?before={before}' if before else '')
                soup = load_page(driver, url, '.tgme_channel_history')
                messages = soup.select('.tgme_widget_message[data-post]')
                if not messages:
                    break
                for message in messages:
                    add(message)
                ids = [int(m['data-post'].split('/')[-1]) for m in messages]
                dates = [read_telegram_message(m)['published_at'][:10] for m in messages]
                following = min(ids)
                if max(dates) < START or following == before:
                    break
                before = following
                time.sleep(PAUSE)
    else:
        seeds = take_sample(list(saved['Telegram'][['channel','post_id']].drop_duplicates().itertuples(index=False,name=None)))
        for channel, post_id in seeds:
            key = f'{channel}/{int(post_id)}'
            soup = load_page(driver, f'https://t.me/s/{key}', '.tgme_channel_history')
            for message in soup.select('.tgme_widget_message[data-post]'):
                add(message, key)
            if key not in rows:
                note_error('telegram', f'https://t.me/{key}', 'Публикация не найдена или больше не отвечает условиям отбора')
            time.sleep(PAUSE)
    return pd.DataFrame(rows.values())

## 4. Пикабу

**Единица:** пост или первоначально загруженный комментарий (`row_type`). Комментарии связаны с постом через `post_id`, ответы между собой — через `parent_comment_id`. Ники и ID авторов отдельно не собираются.

`total_comments` — счётчик сайта во время сбора; `collected_comments` — число действительно сохранённых комментариев внутри периода. В анализе нужны строки соответствующего типа: нельзя суммировать повторяющийся счётчик поста по его комментариям. Теговые и целенаправленно найденные материалы различаются способом обнаружения. Повторная загрузка может дать иной первоначальный комментарий из-за сортировки сайта.

Исходная рамка Пикабу: страницы 1, 11, 21… по шести тематическим тегам с добором соседних и крайних страниц; дополнительные точки в тегах «Высшее образование» и «Рабочие профессии». Точные посещения записаны в протоколе. Квоты до 60 постов на этап служат ориентиром объёма и не создают случайную выборку. `research_screen` — узкий автоматический отбор по заголовку и тексту с явно записанными ручными исключениями; комментарии наследуют признак поста. Поля `manual_review` и `text_available` помогают отделить непроверенные случаи и записи без доступного текста.

In [ ]:
# 4.1. Словари, извлечение одного поста и доступных комментариев
PIKABU_TAGS = {'Колледж': 'выбор СПО и опыт обучения',
 'СПО': 'обсуждение среднего профессионального образования',
 'Высшее образование': 'ценность ВО, критика и защита университета',
 'Рабочие профессии': 'престиж труда и экономические аргументы',
 'Поступление': 'смена образовательных планов при поступлении',
 'ЕГЭ': 'барьеры и стратегии доступа к ВО',
 '10 класс': 'отбор и отказы при переходе в старшую школу',
 '9 класс': 'развилка школа — колледж',
 '11 класс': 'завершение школы и планы поступления',
 'ОГЭ': 'экзамены и отбор после основной школы',
 'ПТУ': 'историческое название и стигма рабочего образования',
 'Техникум': 'альтернативное название организаций СПО',
 'Абитуриент': 'советы, ожидания и пересмотр притязаний',
 'Поступление в вуз': 'конкретные решения и препятствия',
 'Вуз': 'аргументы в пользу и против университетской траектории',
 'Университет': 'ценность диплома и опыт высшего образования',
 'Профориентация': 'советы и институциональное направление выбора',
 'Профессионалитет': 'риторика реформы СПО после 2022 года',
 'Бюджетные места': 'конкуренция и материальная доступность',
 'Платное образование': 'стоимость и вынужденный пересмотр планов'}

PIKABU_CONTEXT = ['колледж', 'техникум', 'вуз', 'университет', 'образован', 'пту', 'спо', 'школ', 'егэ', 'огэ', 'абитуриент',
 'училищ', 'диплом', 'професси', 'класс']

PIKABU_CHOICE = ['поступ', 'прием', 'отбор', 'конкурс', 'зачисл', 'балл', 'бюджет', 'платн', 'стоим', 'отказ', 'не берут',
 'не приняли', 'не прош', 'передум', 'выб', 'совет', 'профориент', 'нужен', 'нужно', 'диплом', 'престиж',
 'успех', 'зарплат', 'зараб', 'востреб', 'трудоустр', 'работ', 'кадр', 'Профессионалитет',
 'Молодые профессионалы', 'WorldSkills', 'EuroSkills', 'разочаров', 'пожалел', 'бесполез', 'ценност',
 'статус', 'мечт', 'хочу', 'хотел', 'застав', 'сокращ', 'доступ', 'возможност', 'перевод', 'перевест']

def _pk_text(node, selector):
    element = node.select_one(selector)
    return element.get_text("\n", strip=True) if element else ""

def _pk_date(node):
    element = node.select_one("time[datetime]")
    return element.get("datetime", "") if element else ""

def _pk_topic(text):
    """Образовательный контекст И выбор/оценка/барьер; знак отношения не задан."""
    return bool(keyword_hits(text, PIKABU_CONTEXT) and keyword_hits(text, PIKABU_CHOICE))

def _pk_url(url):
    parts = urlsplit(url)
    if parts.netloc not in {"pikabu.ru", "www.pikabu.ru"} or not re.fullmatch(
        r"/story/[^/]+_\d+", parts.path
    ):
        return None
    return urlunsplit(("https", "pikabu.ru", parts.path, "", ""))

def parse_pikabu_post(soup, url, config, discovery=None):
    """Один пост + комментарии, уже находящиеся в DOM; кнопки не нажимаются."""
    post_id = url.rsplit("_", 1)[-1]
    post = soup.select_one(f'article.story[data-story-id="{post_id}"]')
    if post is None:
        raise ValueError("На странице нет ожидаемого поста")
    date = _pk_date(post)
    if not date:
        raise ValueError("У поста отсутствует time[datetime]")
    if not within_period(date, config["start"], config["end"]):
        return []

    title = _pk_text(post, ".story__title-link")
    text = "\n\n".join(x.get_text("\n", strip=True) for x in
                       post.select(".story-block_type_text"))
    if not text:
        text = _pk_text(post, ".story__content-inner")
    tags = list(dict.fromkeys(x.get_text(" ", strip=True)
                             for x in post.select(".story__tags a.tags__tag")))
    hits = keyword_hits("\n".join([title, text, *tags]), config["terms"])
    if not hits or not _pk_topic("\n".join([title, text, *tags])):
        return []

    collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    discovery = discovery or {"method": "seed", "tags": []}
    common = dict(
        post_id=post_id, title=title, tags=" | ".join(tags),
        total_comments=parse_count(post.get("data-comments")),
        collected_at=collected_at, discovery_method=discovery["method"],
        discovery_tags=" | ".join(discovery["tags"]),
        comments_scope="первичный DOM без раскрытия; только даты внутри периода",
    )
    comments, seen = [], set()
    for comment in soup.select('.comment[id^="comment_"]'):
        related_post = re.search(r"(?:^|;)sid=(\d+)(?:;|$)", comment.get("data-meta", ""))
        if related_post and related_post.group(1) != post_id:
            continue
        comment_id = comment.get("data-id") or comment["id"].removeprefix("comment_")
        if comment_id in seen:
            continue
        seen.add(comment_id)
        comment_url = f"{url}?cid={comment_id}"
        body = comment.find(class_="comment__body", recursive=False)
        comment_date = _pk_date(body) if body else ""
        if not comment_date:
            note_error("pikabu", comment_url, "Комментарий без даты: не включён")
            continue
        if not within_period(comment_date, config["start"], config["end"]):
            continue
        # pid — родительский КОММЕНТАРИЙ; aid/said — авторы, их не сохраняем.
        parent = re.search(r"(?:^|;)pid=(\d+)(?:;|$)", comment.get("data-meta", ""))
        comment_text = _pk_text(body, ".comment__content")
        comments.append(dict(
            common, row_type="comment", record_id=f"comment:{comment_id}",
            parent_comment_id=parent.group(1) if parent and parent.group(1) != "0" else None,
            url=comment_url, date=comment_date, text=comment_text,
            rating=parse_count(_pk_text(body, ".comment__rating-count")),
            matched_terms=" | ".join(keyword_hits(comment_text, config["terms"])),
        ))
    rows = [dict(
        common, row_type="post", record_id=f"post:{post_id}", parent_comment_id=None,
        url=url, date=date, text=text,
        rating=parse_count(_pk_text(post, ".story__rating-count")),
        matched_terms=" | ".join(hits),
    )] + comments
    for row in rows:
        row["collected_comments"] = len(comments)
    return rows

In [ ]:
# 4.2. Циклы по теговым страницам и публикациям
def pikabu_table(driver):
    urls = list(saved['Пикабу'].loc[lambda d:d.row_type=='post','url'].drop_duplicates())
    if FULL_RUN and EXPAND_ARCHIVE:
        for tag in PIKABU_TAGS:
            old_pages, seen = 0, set()
            for page in range(1, MAX_TAG_PAGES+1):
                soup = load_page(driver, f'https://pikabu.ru/tag/{quote(tag)}?page={page}', 'article.story, #search-header, .stories-feed')
                cards = soup.select('article.story[data-story-id]')
                ids = {c.get('data-story-id') for c in cards if _pk_date(c)}
                if not ids:
                    note_error('pikabu',driver.current_url,'В доступной странице нет датированных карточек; проверьте, действительно ли это конец ленты')
                    break
                if ids <= seen:
                    break
                seen.update(ids)
                dates = [_pk_date(c)[:10] for c in cards if _pk_date(c)]
                old_pages = old_pages+1 if dates and max(dates)<START else 0
                for card in cards:
                    link = card.select_one('.story__title-link[href]')
                    if link and within_period(_pk_date(card),START,END):
                        url = _pk_url(link['href'])
                        if url:
                            urls.append(url)
                if old_pages >= 2:
                    break
                time.sleep(max(3,PAUSE))
    rows = []
    for url in take_sample(list(dict.fromkeys(urls))):
        try:
            soup = load_page(driver,url,'article.story[data-story-id]')
            rows.extend(parse_pikabu_post(soup,url,config,dict(method='archive' if FULL_RUN and EXPAND_ARCHIVE else 'saved_sample_reread',tags=[])))
        except (TimeoutException,ValueError) as error:
            note_error('pikabu',url,str(error))
        time.sleep(max(3,PAUSE))
    return pd.DataFrame(rows).drop_duplicates('record_id') if rows else pd.DataFrame()

## 5. «Табитуриент»

**Единица:** вопрос или ответ; связь — `question_id`. Организация записывается только при явно указанной на сайте приёмной комиссии. Остальные ответы остаются пользовательскими, даже если звучат профессионально.

Исходный отбор: псевдослучайные 400 ID из диапазона до максимального ID, найденного в публичном списке, `seed=42`, затем тематический словарь и проверка дат. Пропуски в нумерации не заменяются вымышленными вопросами. Дополнительно включены проверочные ветки пилота; они маркированы `purposive_seed`. Это выборка из доступных ID, а не всех российских абитуриентов.

Счётчик «Ответов» в ленте может включать вложенные комментарии. Поэтому сохраняются отдельно `replies_reported`, `answers_in_dom` и `nested_comments_reported`. При повторном чтении только страницы ветки `replies_reported` остаётся неизвестным: старый счётчик из готовой базы не переносится как свежий. Сведения из советов не принимаются за проверенные правила поступления.

In [ ]:
# 5.1. Даты, полные тексты и разбор ветки
TABIT_URL = 'https://tabiturient.ru/question/'

TABIT_TERMS = ['СПО', 'колледж', 'техникум', 'ПТУ', 'высшее образован', 'диплом', '10 класс', 'десятый класс', '9 класс',
 'престиж', 'не прой', 'не прош', 'не поступ', 'не взяли', 'не приняли', 'передум', 'попроще', 'отказ',
 'не туда', 'низкие баллы', 'мало баллов', 'платн', 'дорого', 'перевест', 'перевод', 'целево', 'бюджет',
 'профессия', 'профессию', 'нужен ли', 'нужно ли', 'стоит ли', 'работать', 'трудоустрой', 'не потян']

MONTHS = dict(zip('января февраля марта апреля мая июня июля августа сентября октября ноября декабря'.split(), range(1,13)))

def tabit_text(node):
    if node is None:
        return ''
    node = copy(node)
    for label in node.select('[class*="slivmore-pre-"]'):
        label.decompose()  # Убираем «Показать полностью», продолжение уже есть в DOM.
    return node.get_text(' ', strip=True)

def tabit_date(node):
    date = node.select_one('[itemprop="dateCreated"][datetime]')
    if date:
        return date['datetime'][:10]  # Сайт показывает день; псевдоточное время не используем.
    match = re.search(r'(\d{1,2})\s+(' + '|'.join(MONTHS) + r')\s+(20\d{2})', node.get_text(' ', strip=True))
    return f'{match[3]}-{MONTHS[match[2]]:02d}-{int(match[1]):02d}' if match else None

def tabit_cards(soup):
    cards = {}
    for link in soup.select('#resultsliv a[href*="/question/n/?"]'):
        match = re.search(r'/question/n/\?(\d+)$', link.get('href', ''))
        if not match:
            continue
        card = link.find_parent('div', class_='p20')
        if card is None:
            continue
        content = card.find('div', class_='font2', recursive=False)
        answer_link = next((a for a in card.select('a') if 'Ответов' in a.get_text()), None)
        cards[match[1]] = dict(question_id=match[1], url=link['href'], date=tabit_date(card),
                              text=tabit_text(content), tags='; '.join(x.get_text(' ',strip=True) for x in card.select('.button5 b')),
                              replies_reported=parse_count(answer_link.select_one('b').text) if answer_link and answer_link.select_one('b') else None)
    return cards

def parse_tabiturient(soup, card, config):
    qid = card['question_id']
    question = soup.select_one(f'#voprosdivin{qid}')
    if question is None:
        raise ValueError('Не найдена карточка вопроса: изменилась разметка или вопрос удалён')
    qtext = tabit_text(question.select_one('[itemprop="text"]'))
    tags = '; '.join(x.get_text(' ',strip=True) for x in question.select('.button5 b'))
    hits = keyword_hits(qtext + ' ' + tags, config.get('tabit_terms', TABIT_TERMS))
    qdate = tabit_date(question)
    if not qdate:
        raise ValueError('missing_date: у вопроса не найдена дата публикации')
    if not hits or not within_period(qdate,config['start'],config['end']):
        return []
    answers = soup.select('#resultdivcomment > .answ1[id^="commentpad"]')
    rows = []
    for kind, node in [('question',question)] + [('answer',a) for a in answers]:
        ident = qid if kind == 'question' else re.search(r'(\d+)$',node['id'])[1]
        date = tabit_date(node)
        if not date:
            note_error('tabiturient',card['url'],f'{kind} {ident}: дата отсутствует')
            continue
        if not within_period(date,config['start'],config['end']):
            continue
        author = node.select_one('[itemprop="author"]')
        author_text = author.get_text(' ',strip=True) if author else ''
        university = bool(re.search(r'при[её]мн\w* комисси',author_text,re.I))
        comments_link = node.select_one('a[onclick^="showdopcomment"] b')
        vote = node.select_one('[itemprop="upvoteCount"]')
        rows.append(dict(record_id=f'{kind}_{ident}', row_type=kind, question_id=qid,
                         answer_id=ident if kind=='answer' else None,
                         url=card['url'] + (f'#commentpad{ident}' if kind=='answer' else ''),
                         date=date, text=tabit_text(node.select_one('[itemprop="text"]')),
                         tags=tags, author_type='university_label' if university else ('question_author' if kind=='question' else 'user'),
                         organization=author_text if university else None, likes=parse_count(vote.text) if vote else None,
                         replies_reported=card.get('replies_reported'), answers_in_dom=len(answers),
                         nested_comments_reported=parse_count(comments_link.text) if comments_link else None,
                         matched_terms='; '.join(hits),
                         collected_at=datetime.now(MOSCOW).isoformat(timespec='seconds'),
                         comments_scope='answers in initial DOM; nested comments not loaded'))
    return rows

In [ ]:
# 5.2. Цикл по вопросам; ответы сохраняются вместе с вопросом
def tabiturient_table(driver):
    questions = saved['Табитуриент'].loc[lambda d:d.row_type=='question','question_id'].drop_duplicates()
    ids = [str(int(i)) for i in questions]
    if FULL_RUN and EXPAND_ARCHIVE:
        soup = load_page(driver,TABIT_URL,'#resultsliv a[href*="/question/n/?"]')
        last_id = max(map(int,tabit_cards(soup)))
        ids = [str(i) for i in range(1,last_id+1)]
    rows = []
    for question_id in take_sample(ids):
        url = f'{TABIT_URL}n/?{question_id}'
        soup = load_page(driver,url,'body')
        if not soup.select_one(f'#voprosdivin{question_id}'):
            note_error('tabiturient',url,'Вопрос отсутствует в доступной странице')
            continue
        items = parse_tabiturient(soup,dict(question_id=question_id,url=url),config)
        for row in items:
            row['selection_method'] = 'archive' if FULL_RUN and EXPAND_ARCHIVE else 'saved_sample_reread'
        rows.extend(items)
        time.sleep(PAUSE)
    return pd.DataFrame(rows).drop_duplicates('record_id') if rows else pd.DataFrame()

## 6. Выполнение и проверка

Каждый источник открывается в отдельной короткой браузерной сессии; сессия закрывается и при ошибке. В ходе выполнения сохраняются CSV. Если источник изменит структуру, выполнение остановится с ошибкой, а уже записанные таблицы останутся в папке запуска. Для многочасового обхода рекомендуется версия со страницами состояния из папки `scripts`.

Сводка ниже относится **к текущему повторному сбору**. Готовая выборка из приложенного Excel показана отдельно в начале notebook. Небольшая демонстрация не является полной исследовательской базой.

In [ ]:
# 6.1. Производные поля и проверки качества
FOCUS_CONTEXT = ['колледж', 'техникум', 'СПО', 'ПТУ', 'вуз', 'университет', 'высшее образован', '10 класс', 'десятый класс',
 'профориент', 'Профессионалитет', 'рабочие профессии']

FOCUS_CHOICE = ['поступ', 'приём', 'прием', 'выбор', 'выбира', 'выбрал', 'отбор', 'отказ', 'не берут', 'не приняли', 'балл',
 'бюджет', 'платн', 'диплом', 'престиж', 'передум', 'стоит', 'нужен', 'нужно', 'зарплат', 'зарабат', 'успех',
 'карьер', 'востреб', 'трудоустр', 'дефицит', 'профориент', 'хочу', 'хотел', 'возможност', 'доступ',
 'професси', 'перевод', 'перевест']

SCREEN_CONTEXT = ['колледж', 'техникум', 'вуз', 'университет', 'образован', 'ПТУ', 'СПО', 'школ', 'ЕГЭ', 'ОГЭ', 'абитуриент',
 'училищ', 'диплом', '10 класс', '10-й класс', 'десятый класс']

SCREEN_CHOICE = ['поступ', 'вступительн', 'приемная комисс', 'приёмная комисс', 'отбор', 'зачисл', 'абитуриент', 'профориент',
 'диплом', 'престиж', 'трудоустрой', 'востребованност', 'рабочие профессии', 'Профессионалитет',
 'бюджетные места', 'платное обучение', 'платное образование', 'конкурс аттестатов', 'не берут в',
 'не взяли в', 'не приняли в', 'после 9', 'после девятого', '10 класс', '10-й класс', 'десятый класс',
 'после колледжа', 'вуз после', 'из колледжа в', 'из вуза в', 'перевестись в', 'перевод в вуз',
 'перевод в колледж', 'выбор профессии', 'смена профессии', 'ценность образования', 'образование не нужно',
 'образование нужно', 'вуз не нужен', 'нужен диплом']

MANUAL_EXCLUDE = {'10641449': 'Запрос названия американской комедии; образовательный выбор не обсуждается',
 '6355320': 'Бытовая история о телефоне и конфликте в семье; образование лишь в тегах',
 '6360292': 'Анекдот о коньяке преподавателя; не об образовательной траектории',
 '9080062': 'Реклама волонтёрской работы; колледж только в тегах'}

PILOT_RELEVANT = {'5622243', '14388561', '10410436', '12639540', '11554177', '11441073'}

def prepare_frame(source, frame):
    frame = frame.copy()
    if source == 'telegram':
        frame['record_id'] = frame['channel'].astype(str) + '/' + frame['post_id'].astype(str)
        frame['row_type'] = 'post'
    assert frame.columns.is_unique, f'{source}: повторяются столбцы'
    assert not frame['record_id'].duplicated().any(), f'{source}: повторяются ID'
    assert not frame.duplicated().any(), f'{source}: повторяются строки'
    date_column = 'published_at' if source == 'telegram' else 'date'
    assert frame[date_column].map(lambda x:within_period(x,'2018-01-01','2026-10-06')).all(), f'{source}: дата вне периода'
    dates = frame[date_column].map(lambda x:datetime.fromisoformat(str(x).replace('Z','+00:00')))
    frame['date_local'] = dates.map(lambda d:(d.astimezone(MOSCOW) if d.tzinfo else d).date().isoformat())
    frame['year'] = frame['date_local'].str[:4].astype(int)
    frame['period'] = frame['year'].map(lambda y:'2018' if y==2018 else ('2019–2021' if y<=2021 else '2022–2026'))
    frame['hashtags'] = frame['text'].fillna('').map(lambda s:'; '.join(dict.fromkeys(re.findall(r'(?<!\w)#\w+',s))))
    frame['text_available'] = frame['text'].fillna('').str.strip().str.len().gt(0)
    # Это признак для сортировки кандидатов; он не заменяет ручное кодирование охлаждения.
    context = frame['text'].fillna('') + ' ' + frame.get('title',pd.Series('',index=frame.index)).fillna('')
    context = context.map(lambda s:' '.join(s.split()))
    frame['focus_candidate'] = context.map(lambda s:bool(keyword_hits(s,FOCUS_CONTEXT) and keyword_hits(s,FOCUS_CHOICE)))
    if source == 'pikabu':
        posts = frame[frame.row_type.eq('post')]
        screen,review = {},{}
        for _,row in posts.iterrows():
            pid = str(row.post_id)
            text = ' '.join((str(row.title) + ' ' + str(row.text)).split())  # Теги не подменяют содержание.
            screen[pid] = bool(keyword_hits(text,SCREEN_CONTEXT) and keyword_hits(text,SCREEN_CHOICE))
            review[pid] = 'not_reviewed'
            if pid in MANUAL_EXCLUDE:
                screen[pid],review[pid] = False,'excluded: '+MANUAL_EXCLUDE[pid]
            elif pid in PILOT_RELEVANT:
                screen[pid],review[pid] = True,'included: проверен в предварительном пилоте'
        frame['research_screen'] = frame['post_id'].astype(str).map(screen)
        frame['manual_review'] = frame['post_id'].astype(str).map(review)
    if source == 'telegram':
        snapshots = pd.to_datetime(frame['collected_at'],utc=True)
        publications = pd.to_datetime(frame['published_at'],utc=True)
        frame['age_days_at_collection'] = (snapshots-publications).dt.total_seconds().div(86400).round(2)
        frame['at_least_14_days'] = frame['age_days_at_collection'] >= 14
    leading = ['record_id','row_type','date_local','period']
    leading += [c for c in ['channel','title','text','url'] if c in frame]
    frame = frame[leading + [c for c in frame.columns if c not in leading]]
    return frame.sort_values(['date_local','record_id']).reset_index(drop=True)

def validate_relations(tables):
    pikabu = tables['Пикабу']
    assert set(pikabu.loc[pikabu.row_type=='comment','post_id']) <= set(pikabu.loc[pikabu.row_type=='post','post_id'])
    tabit = tables['Табитуриент']
    assert set(tabit.loc[tabit.row_type=='answer','question_id']) <= set(tabit.loc[tabit.row_type=='question','question_id'])
    return pd.DataFrame([dict(source=name,rows=len(df),units=df['row_type'].value_counts().to_dict(),
                              first_date=df.date_local.min(),last_date=df.date_local.max(),focus_candidates=int(df.focus_candidate.sum()))
                         for name,df in tables.items()])

In [ ]:
# 6.2. Selenium → список словарей → DataFrame → проверки
tables = {}
for source, sheet, collector in [('telegram','Telegram',telegram_table),('pikabu','Пикабу',pikabu_table),('tabiturient','Табитуриент',tabiturient_table)]:
    driver = create_driver()
    try:
        frame = collector(driver)
    finally:
        driver.quit()
    if frame.empty:
        raise ValueError(f'{sheet}: пустая выгрузка, проверьте доступность и журнал')
    tables[sheet] = prepare_frame(source,frame)
    suffix = 'repeat' if FULL_RUN else 'demo'
    tables[sheet].to_csv(f'{source}_{suffix}.csv',index=False,encoding='utf-8-sig')
# Вывод: объём, типы сообщений и реальные временные границы свежего сбора.
display(validate_relations(tables))

## 7. Экспорт

В Excel одна строка соответствует одному сообщению. Полный текст хранится в ячейке; высота строк компактная, для чтения длинного текста можно увеличить её или открыть строку формул. Если текст превышает предел Excel, продолжение переносится в `text_part_2`, `text_part_3` и т. д.; CSV сохраняет его единым полем. Текст сайта не исполняется как формула.

Исходный файл `База_трёх_источников.xlsx` не перезаписывается. Демо сохраняется как `Демонстрация.xlsx`, повторный сбор — как `Повторный_сбор.xlsx`.

In [ ]:
# 7. Запись трёх таблиц в один Excel
def save_excel(tables, path):
    """Три базы на трёх листах, сохранение pandas. Полный текст остаётся в CSV."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        for name, frame in tables.items():
            frame = frame.copy()
            if 'date_local' in frame:
                frame['date_local'] = pd.to_datetime(frame['date_local'])
            for column in list(frame.select_dtypes(include=['object', 'string'])):
                # Удаляем только недопустимые управляющие символы Excel, не слова.
                frame[column] = frame[column].map(lambda x: re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', x) if isinstance(x,str) else x)
                maximum = frame[column].map(lambda x: len(x) if isinstance(x,str) else 0).max()
                if pd.notna(maximum) and maximum > 32767:
                    for part in range(1,(int(maximum)-1)//32000+1):
                        frame[f'{column}_part_{part+1}'] = frame[column].map(lambda x: x[part*32000:(part+1)*32000] if isinstance(x,str) else x)
                    frame[column] = frame[column].map(lambda x:x[:32000] if isinstance(x,str) else x)
            frame.to_excel(writer, sheet_name=name, index=False)
            sheet = writer.sheets[name]
            sheet.freeze_panes = 'C2'
            sheet.auto_filter.ref = sheet.dimensions
            from openpyxl.styles import Alignment, Font, PatternFill
            sheet.sheet_view.showGridLines = False
            sheet.row_dimensions[1].height = 32
            for cell in sheet[1]:
                cell.fill = PatternFill('solid',fgColor='17324D')
                cell.font = Font(color='FFFFFF',bold=True)
                cell.alignment = Alignment(wrap_text=True,vertical='center')
            for row in sheet.iter_rows(min_row=2):
                sheet.row_dimensions[row[0].row].height = 48
                for cell in row:
                    if isinstance(cell.value,str):
                        cell.data_type = 's'  # Текст сайта, начинающийся с '=', не формула.
                    cell.alignment = Alignment(vertical='top',wrap_text=True)
                    if frame.columns[cell.column-1] == 'date_local':
                        cell.number_format = 'dd.mm.yyyy'
                    if cell.row % 2 == 0:
                        cell.fill = PatternFill('solid',fgColor='F1F5F9')
            for col in sheet.columns:
                label = str(col[0].value)
                sheet.column_dimensions[col[0].column_letter].width = 75 if label in ['text','title'] else (42 if label=='url' else min(35,max(15,len(label)+2)))
output_name = 'Повторный_сбор.xlsx' if FULL_RUN else 'Демонстрация.xlsx'
save_excel(tables,output_name)
# Вывод подтверждает имя созданного файла, а не полноту охвата сайтов.
print(f'Сохранено: {output_name}')

## Словарь переменных

Все перечисленные поля относятся к шагам 3–7. Пустая ячейка — отсутствие доступного значения, а не ноль.

| Поле | Значение |
|---|---|
| `record_id` | Уникальный ключ сообщения: канал/ID в Telegram, тип:ID на Пикабу, тип_ID на Табитуриенте. Это ID контента, не автора. |
| `row_type` | post — пост; comment — комментарий Пикабу; question — вопрос; answer — ответ Табитуриента. |
| `date_local` | Календарная дата публикации по Москве; в Excel — дата, в CSV — YYYY-MM-DD. |
| `period` | Этап: 2018, 2019–2021 или 2022–2026. Не является оценкой влияния реформ. |
| `year` | Год публикации, вычисленный из date_local. |
| `channel` | Публичное имя официального канала Telegram. |
| `title` | Заголовок родительского поста Пикабу; повторяется у его комментариев для контекста. |
| `text` | Полный доступный текст данной единицы из DOM; скрытое продолжение ответа включено. Текст изображений и видео не распознавался. |
| `url` | Постоянная ссылка на источник сообщения; для ответа/комментария — с идентификатором или якорем. |
| `post_id` | ID поста. В Telegram уникален только вместе с channel; на Пикабу связывает пост и комментарии. |
| `published_at` | Исходный момент публикации Telegram в ISO 8601 с московским смещением. |
| `date` | Дата/момент сообщения Пикабу либо день сообщения Табитуриента; точность источников различается. |
| `views_raw` | Просмотры как показаны Telegram, например 1.54K; пропуск означает недоступность. |
| `views` | Числовое приближение views_raw. K/M на сайте уже округлены. |
| `reactions_raw` | JSON-массив исходных подписей доступных реакций; пропуск не приравнивается к нулю. |
| `reactions` | JSON-массив пар reaction/count. Знак отношения и валентность здесь не кодируются. |
| `media` | Виды обнаруженных вложений Telegram, разделённые точкой с запятой. |
| `media_count` | Число элементов медиа в карточке DOM; не число просмотров или пересылок. |
| `album_post_ids` | JSON-массив ID вложенных сообщений альбома. Альбом сохранён одной строкой. |
| `is_forward` | Наличие блока пересылки в карточке Telegram; количество пересылок недоступно. |
| `query_matches` | Поисковые запросы, через которые найден пост в поисковом режиме; в архивном отборе пусто. |
| `sample_windows` | Окна сезонной выборки Telegram, в которые попала публикация. Повторно найденный пост не дублируется. |
| `selection_method` | Способ отбора: random_id_seed42 — случайный ID; purposive_seed — целевое дополнение; при повторном запуске saved_sample_reread или archive. |
| `matched_terms` | Сработавшие слова/основы слов. У ответов Табитуриента — условия отбора родительского вопроса; у комментария Пикабу — совпадения в самом комментарии. |
| `collected_at` | Момент чтения страницы. Значения просмотров, реакций и оценок относятся к этому моменту. |
| `hashtags` | Хэштеги, извлечённые из текста без удаления из исходного поля text. |
| `text_available` | Есть непустой извлечённый текст данной единицы. False не означает отсутствие смысла в изображении или видео: OCR и расшифровка не выполнялись. |
| `focus_candidate` | Вспомогательное совпадение образовательного контекста и лексики выбора/оценки. Нужна ручная проверка; это не код охлаждения. |
| `research_screen` | Более узкий отбор Пикабу по title+text: образовательный контекст и специальная лексика выбора/ценности. Учтены явные ручные исключения и проверенные случаи пилота; комментарии наследуют значение поста. Это не окончательная содержательная разметка. |
| `manual_review` | Пикабу: not_reviewed — вручную не оценено; included — проверено в пилоте; excluded — указанная причина ручного исключения. Значение относится к ветке целиком. |
| `age_days_at_collection` | Возраст поста Telegram в сутках в момент снимка; рассчитан по точным датам. |
| `at_least_14_days` | Возраст Telegram-поста не менее 14 суток; признак для будущего анализа вовлечённости, не условие удаления. |
| `parent_comment_id` | ID родительского комментария Пикабу, если есть. Родитель может не входить в первоначально загруженный фрагмент. |
| `tags` | Теги публикации Пикабу или предметы/темы вопроса Табитуриента. У ответов повторён контекст вопроса. |
| `rating` | Рейтинг данного поста/комментария Пикабу во время сбора. Может быть отрицательным. |
| `total_comments` | Счётчик всех комментариев поста Пикабу на сайте; повторён у связанных строк и не подлежит суммированию по ним. |
| `collected_comments` | Число фактически сохранённых комментариев данного поста Пикабу внутри периода; повторено у связанных строк. |
| `discovery_method` | Как найден пост Пикабу: теговая лента, систематически выбранные страницы с добором или целевой URL пилота. |
| `discovery_tags` | Теги, через которые найден пост; отличаются от всех тегов самого поста. |
| `comments_scope` | Явное описание доступного фрагмента обсуждения и ограничений подгрузки. |
| `question_id` | ID вопроса Табитуриента, связывающий вопрос со всеми включёнными ответами. |
| `answer_id` | ID ответа Табитуриента; пусто для строки вопроса. |
| `author_type` | question_author — автор вопроса; university_label — явная метка приёмной комиссии; user — другой автор ответа без такой метки. |
| `organization` | Отображаемое название и метка приёмной комиссии. Для обычных пользователей имя не сохраняется. |
| `likes` | Показанное число отметок Полезно у конкретного вопроса или ответа Табитуриента. |
| `replies_reported` | Счётчик Ответов из общей ленты Табитуриента, если он был прочитан. Может включать вложенные комментарии; пропуск не равен нулю. |
| `answers_in_dom` | Количество карточек верхнеуровневых ответов в загруженной ветке, до временного фильтра; повторяется у строк ветки. |
| `nested_comments_reported` | Число вложенных комментариев, показанное рядом с конкретным ответом; сами комментарии не загружались. |
| `text_part_N` | При необходимости — последовательные продолжения текста сверх лимита Excel. В CSV исходный текст не разбивается. |

## Результат исходного сбора 07.10.2026

| Источник | Строки | Единицы наблюдения | Первое сообщение | Последнее сообщение |
|---|---:|---|---|---|
| Telegram | 420 | посты: 420 | 2020-07-17 | 2026-10-06 |
| Пикабу | 340 | посты: 174; комментарии: 166 | 2018-01-10 | 2026-10-04 |
| Табитуриент | 334 | ответы: 205; вопросы: 129 | 2020-10-16 | 2026-09-22 |

Распределение **собранных сообщений**, включая связанные ответы и комментарии. Это характеристика выборки, а не частота мнений в генеральной совокупности.

| Этап | Telegram | Пикабу | Табитуриент |
|---|---:|---:|---:|
| 2018 | 0 | 107 | 0 |
| 2019–2021 | 60 | 118 | 47 |
| 2022–2026 | 360 | 115 | 287 |

У выбранных Telegram-витрин не наблюдается публикаций за 2018–2019; в выборке Табитуриента ранняя доступная дата начинается в 2020 году. Значения 0 здесь означают отсутствие **собранных** единиц; полноценную межплатформенную динамику с 2018 года по ним построить нельзя.

На «Табитуриенте» только 7 из 205 ответов имеют явную метку приёмной комиссии; они относятся к двум организациям. Остальные ответы нельзя использовать как высказывания вузов. Это ограничивает проверку институциональных стратегий на данном источнике.


## Зафиксированные затруднения и ограничения

- **Поиск Telegram оказался неполным.** Перешли к чтению обычной ленты. Первый доступный пост Минпросвещения — 07.04.2022; Минобрнауки — 16.03.2020. Это границы наблюдаемой витрины, а не доказательство отсутствия прежней государственной риторики.
- **Старые периоды неодинаково представлены.** Нулевую ячейку покрытия нельзя заменять вымышленной публикацией; источник не позволяет построить сбалансированную панель за 2018–2026.
- **Редкая сетка Пикабу пропускала ранние публикации.** Дополнительно проверены соседние страницы и края доступных лент; точные страницы сохранены в протоколе. Пустая страница и счётчик теговой ленты не приняты за доказательство полного охвата архива.
- **Синхронизация папки временно блокировала промежуточный файл.** В длительном парсере Пикабу добавлены ограниченные повторы записи; продолжение использовало уже сохранённые страницы.
- **Дата видео отличается от даты поста.** В Telegram выбирается дата постоянной ссылки, а не длительность ролика. В Пикабу используется `time[datetime]`, а не ненадёжный числовой атрибут карточки.
- **Длинные ответы были визуально свёрнуты.** Читаем весь DOM и убираем только служебную кнопку; продолжение не теряется и не дублируется.
- **Комментарии доступны не полностью.** На Пикабу — первоначальный DOM; на «Табитуриенте» — ответы без загрузки вложенных комментариев; в Telegram комментарии не собираются. Эти ограничения сохраняются в полях.
- **Дата публикации и дата измерения счётчика различаются.** Публикации/ответы обрезаны 06.10.2026; просмотры, оценки и реакции измерены при сборе 07.10.2026. Восстановить их значение на конец 06.10 по текущей странице нельзя. Посты моложе 14 дней сохранены, но выделены для будущего анализа вовлечённости.
- **Отбор словарём широкий.** `focus_candidate` помогает просматривать тексты, но ручная проверка остаётся обязательной. Нельзя заранее приписывать всем совпадениям «охлаждение», а всем эмодзи — согласие с политикой.
- **Надёжность проверялась на живых страницах и граничных случаях:** переход московской даты, пропуск против нуля, полный скрытый ответ, метка приёмной комиссии, различение ответа и комментария, привязка сообщения к родителю. Перезапуск всех ячеек проверен отдельно; результаты проверки приложены к проекту.

Следующий содержательный этап: проверка связи с российским образовательным контекстом, уточнение релевантности, двойная ручная разметка стратегий и направлений переориентации; только затем оценка согласованности кодировщиков и анализ изменений. Для оценки доли тем во всей коммуникации требуется расширенная рамка отбора с соответствующим знаменателем.